In [4]:
from pathlib import Path

print("Local preprocessing setup ready.")

Local preprocessing setup ready.


In [5]:

from pathlib import Path

INPUT_ROOT = Path(
    r"C:\Users\dev20\OneDrive\Desktop\THE END\MLP\MLP-Dataset"
)

OUTPUT_ROOT = Path(
    r"C:\Users\dev20\OneDrive\Desktop\THE END\MLP\GaitTensors1"
)

split_aliases = {
    "train": ["train", "Training"],
    "validation": ["Validation", "val"],
    "test": ["Test", "testing"]
}

print("Input folder exists:", INPUT_ROOT.exists())
print("Output folder:", OUTPUT_ROOT)


Input folder exists: True
Output folder: C:\Users\dev20\OneDrive\Desktop\THE END\MLP\GaitTensors1


In [6]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F

NUM_FRAMES = 32
WALKING_SECONDS = 7.0
SIDE_SIZE = 256
CROP_SIZE = 256

MEAN = torch.tensor([0.45, 0.45, 0.45]).view(1, 3, 1, 1)
STD = torch.tensor([0.225, 0.225, 0.225]).view(1, 3, 1, 1)

VIDEO_EXTENSIONS = {
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".mpeg",
    ".mpg",
    ".m4v"
}

def preprocess_video(video_path):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError("Could not open video")

    fps = cap.get(cv2.CAP_PROP_FPS)
    original_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    original_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    original_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    decoded_frames = []

    try:
        while True:
            ok, frame_bgr = cap.read()
            if not ok:
                break

            frame_rgb = cv2.cvtColor(
                frame_bgr, cv2.COLOR_BGR2RGB
            )
            decoded_frames.append(frame_rgb)
    finally:
        cap.release()

    if len(decoded_frames) == 0:
        raise RuntimeError("No frames decoded")

    # [T, H, W, C]
    video_array = np.stack(decoded_frames)

    if fps <= 0:
        raise RuntimeError("Invalid FPS reported by video")


    # Select the first 32 frames
    walking_frames = 32

    walking_video = video_array[:walking_frames]
    indices = np.arange(0,32).astype(np.int64)

    video_array = walking_video[indices]

    # [T, H, W, C] -> [T, C, H, W], scale to [0, 1].
    frames = torch.from_numpy(video_array).permute(
        0, 3, 1, 2
    ).float() / 255.0

    # Resize while preserving the aspect ratio.
    _, _, H, W = frames.shape

    scale = SIDE_SIZE / max(H, W)
    new_h = max(1, round(H * scale))
    new_w = max(1, round(W * scale))

    frames = F.interpolate(
        frames,
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    )

    # Pad to 256 x 256 without cropping any content.
    _, _, H, W = frames.shape

    pad_top = (CROP_SIZE - H) // 2
    pad_bottom = CROP_SIZE - H - pad_top
    pad_left = (CROP_SIZE - W) // 2
    pad_right = CROP_SIZE - W - pad_left

    frames = F.pad(
        frames,
        (pad_left, pad_right, pad_top, pad_bottom),
        mode="constant",
        value=0
    )

    if frames.shape[-2:] != (256, 256):
        raise RuntimeError(
            f"Unexpected frame size: {frames.shape}"
        )

    # Normalize RGB channels.
    frames = (frames - MEAN) / STD

    # [T, C, H, W] -> [C, T, H, W]
    frames = frames.permute(1, 0, 2, 3).contiguous()

    metadata = {
        "fps": fps,
        "decoded_frames": len(decoded_frames),
        "sampled_frames": len(indices),
        "walking_interval_seconds": walking_frames / fps,
        "original_width": original_width,
        "original_height": original_height,
        "final_height": 256,
        "final_width": 256,
        "padding_used": True
    }

    return frames, metadata

In [9]:

split_paths = {}

for split, aliases in split_aliases.items():
    for alias in aliases:
        candidate = INPUT_ROOT / alias

        if candidate.is_dir():
            split_paths[split] = candidate
            break

    if split not in split_paths:
        raise FileNotFoundError(
            f"Missing folder for {split}: {aliases}"
        )

print("Detected dataset folders:")
for split, folder in split_paths.items():
    print(split, ":", folder)


Detected dataset folders:
train : C:\Users\dev20\OneDrive\Desktop\THE END\MLP\MLP-Dataset\Training
validation : C:\Users\dev20\OneDrive\Desktop\THE END\MLP\MLP-Dataset\Validation
test : C:\Users\dev20\OneDrive\Desktop\THE END\MLP\MLP-Dataset\Test


In [11]:
import time
import csv
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

log_path = OUTPUT_ROOT / "preprocessing_log1.csv"

if log_path.exists():
    with open(log_path, newline="", encoding="utf-8") as f:
        old_log = list(csv.DictReader(f))
else:
    old_log = []

# Keep existing log entries when resuming.
log_by_source = {
    row["source"]: row
    for row in old_log
}

# Only skip a file if its output exists and its previous
# processing was recorded as successful.
success_sources = {
    source for source, row in log_by_source.items()
    if row["status"] == "success"
}

video_paths = []

for split, split_dir in split_paths.items():
    for class_name in ["Regular", "Limping", "Shuffling"]:
        class_dir = split_dir / class_name

        if not class_dir.is_dir():
            raise FileNotFoundError(
                f"Missing class folder: {class_dir}"
            )

        for path in sorted(class_dir.rglob("*")):
            if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS:
                video_paths.append(
                    (split, split_dir, class_name, path)
                )

print("Videos found:", len(video_paths))

if len(video_paths) != 899:
    print(
        "WARNING: Expected 1919 videos. "
        "Check the folder structure and file extensions."
    )

start_time = time.time()
processed = 0
failed = 0

for i, (split, split_dir, class_name, video_path) in enumerate(
    video_paths, start=1
):
    # Preserve any subfolders and the original filename.
    relative_path = video_path.relative_to(split_dir)
    output_path = (
        OUTPUT_ROOT / split / relative_path
    ).with_suffix(".pt")

    source_key = str(video_path)

    if (
        source_key in success_sources
        and output_path.exists()
    ):
        processed += 1
        continue

    output_path.parent.mkdir(parents=True, exist_ok=True)

    row = {
        "source": source_key,
        "output": str(output_path),
        "split": split,
        "class": class_name,
        "status": "failed",
        "error": "",
        "shape": "",
        "fps": "",
        "decoded_frames": "",
        "original_width": "",
        "original_height": ""
    }

    try:
        tensor, meta = preprocess_video(video_path)

        # Save only the normalized video tensor.
        torch.save(tensor, output_path)

        row.update({
            "status": "success",
            "shape": str(list(tensor.shape)),
            "fps": meta["fps"],
            "decoded_frames": meta["decoded_frames"],
            "original_width": meta["original_width"],
            "original_height": meta["original_height"]
        })

        processed += 1

    except Exception as e:
        row["error"] = repr(e)
        failed += 1

    log_by_source[source_key] = row

    # Save the log periodically so progress survives interruptions.
    if i % 20 == 0 or i == len(video_paths):
        with open(log_path, "w", newline="", encoding="utf-8") as f:
            fieldnames = list(next(iter(log_by_source.values())).keys())
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(log_by_source.values())

    if i % 10 == 0 or i == len(video_paths):
        print(
            f"[{i}/{len(video_paths)}] "
            f"Successful this run: {processed} | "
            f"Failed this run: {failed}"
        )

# Ensure the final log is saved.
with open(log_path, "w", newline="", encoding="utf-8") as f:
    fieldnames = list(next(iter(log_by_source.values())).keys())
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(log_by_source.values())

print("\nPreprocessing finished.")
print("Output folder:", OUTPUT_ROOT)
print("Log:", log_path)
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))
print("Failed this run:", failed)

Videos found: 899

Preprocessing finished.
Output folder: C:\Users\dev20\OneDrive\Desktop\THE END\MLP\GaitTensors1
Log: C:\Users\dev20\OneDrive\Desktop\THE END\MLP\GaitTensors1\preprocessing_log1.csv
Elapsed minutes: 0.0
Failed this run: 1


In [12]:
from collections import Counter

results = list(log_by_source.values())

print("Total logged:", len(results))
print("Successful:", sum(r["status"] == "success" for r in results))
print("Failed:", sum(r["status"] == "failed" for r in results))

for split in split_paths:
    for class_name in ["Regular", "Limping", "Shuffling"]:
        folder = OUTPUT_ROOT / split / class_name
        files = list(folder.rglob("*.pt")) if folder.exists() else []
        print(f"{split}/{class_name}: {len(files)} tensors")

# Load and verify one saved tensor.
sample_files = list(OUTPUT_ROOT.rglob("*.pt"))

if sample_files:
    sample = torch.load(
        sample_files[0],
        map_location="cpu",
        weights_only=True
    )
    print("Sample file:", sample_files[0])
    print("Tensor shape:", sample.shape)
    print("Tensor dtype:", sample.dtype)
    print("All finite:", torch.isfinite(sample).all().item())

Total logged: 899
Successful: 898
Failed: 1
train/Regular: 199 tensors
train/Limping: 200 tensors
train/Shuffling: 200 tensors
validation/Regular: 50 tensors
validation/Limping: 50 tensors
validation/Shuffling: 50 tensors
test/Regular: 51 tensors
test/Limping: 49 tensors
test/Shuffling: 49 tensors
Sample file: C:\Users\dev20\OneDrive\Desktop\THE END\MLP\GaitTensors1\test\Limping\Instance_1770.0.pt
Tensor shape: torch.Size([3, 32, 256, 256])
Tensor dtype: torch.float32
All finite: True
